In [40]:
import customfunctions as ctf
import pandas as pd
import numpy as np
import heartpy as hp
import neurokit2 as nk
import prepro as prep  # Custom functions for preprocessing
import os
import glob
from pathlib import Path
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Parameters

### Preprocessing parameters

In [41]:
# BVP preprocessing
bvp_prepro_winsz = 10
bvp_prepro_ovlap = 0.02
bvp_fs = 64

# EDA preprocessing
eda_fs = 4
iter_num = 7

### Processing parameters

In [42]:
basalscene = 0
targetscene = 3
# Note, subject 10 does not have scene 2 nor 3 data
subjects = [1, 2, 3, 4, 6, 8, 9]
winsz = 90
ovlap = 0.5
# Outlier removal method, either iqr_outlier, mod_zscore_outlier (both applicable in non-normal data) or winsorization (winsor_outlier)
outliermeth = 'quant_outlier'
# Outlier treatment strategy, either elimination ('elim') or imputation ('impute')
outliertreatment = 'elim'
p_low=0.5
p_high=0.95
# Normalization type, either z-score ('zscore'), mean ('mean'), minmax scaling ('minmax'), robust scaling ('robust')
normtype = 'minmax'
base_dir = Path().resolve()

# Data import function

In [43]:
folderpath = os.path.join(base_dir, "..", "RawData")
folderpath = os.path.abspath(folderpath)


def dataimport(subjects, folderpath, scene):
    bvp_rawdata_list = []
    eda_rawdata_list = []

    for subject in subjects:
        subject_str = f"S{subject}" if subject == 10 else f"S{subject:02d}"
        pattern = os.path.join(
            folderpath,
            f"S{subject}",
            "Empatica",
            f"P300_{subject_str}R0{scene}*",
            "Raw",
        )
        matched_folders = glob.glob(pattern)
        raw_folder = matched_folders[0]
        bvp_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileBVP.csv")).drop(
            "Datetime", axis=1
        )
        eda_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileEDA.csv")).drop(
            "Datetime", axis=1
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_rawdata_list.append(bvp_dataholder)
        eda_rawdata_list.append(eda_dataholder)

    bvp_rawdata_df = pd.concat(bvp_rawdata_list, ignore_index=True)
    eda_rawdata_df = pd.concat(eda_rawdata_list, ignore_index=True)

    return bvp_rawdata_df, eda_rawdata_df


basal_bvp_rawdata, basal_eda_rawdata = dataimport(
    subjects, folderpath, scene=basalscene
)
tscene_bvp_rawdata, tscene_eda_rawdata = dataimport(
    subjects, folderpath, scene=targetscene
)

# Preprocessing function

In [44]:
def preprocess(subjects, bvp_df, eda_df, bvp_fs, eda_fs, winsz, ovlap, iter_num):
    bvp_prepdata_list = []
    eda_prepdata_list = []
    for subject in subjects:
        bvp_dataholder = pd.DataFrame(
            prep.preprocess_bvp(
                sig=bvp_df[bvp_df["Subject"] == subject]["valueBVP"],
                fs=bvp_fs,
                winsz=winsz,
                ovlap=ovlap,
            ),
            columns=["valueBVP"],
        )
        eda_dataholder = pd.DataFrame(
            prep.preprocess_eda(
                sig=eda_df[eda_df["Subject"] == subject]["valueEDA"],
                fs=eda_fs,
                iter_num=iter_num,
                verbose=False,
            )[0],
            columns=["valueEDA"],
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_prepdata_list.append(bvp_dataholder)
        eda_prepdata_list.append(eda_dataholder)

    bvp_prepdata_df = pd.concat(bvp_prepdata_list, ignore_index=True)
    eda_prepdata_df = pd.concat(eda_prepdata_list, ignore_index=True)

    return bvp_prepdata_df, eda_prepdata_df


basal_bvp_prepdata, basal_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=basal_bvp_rawdata,
    eda_df=basal_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)

tscene_bvp_prepdata, tscene_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=tscene_bvp_rawdata,
    eda_df=tscene_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)


# Processing function

In [45]:
import warnings
def get_bvp_feats(window, smooth_window, fs):
    wd = {}
    wd = hp.peakdetection.detect_peaks(
        window, smooth_window, ma_perc=20, sample_rate=fs
    )
    wd = hp.analysis.calc_rr(wd["peaklist"], sample_rate=fs, working_data=wd)
    wd = hp.peakdetection.check_peaks(
        wd["RR_list"], wd["peaklist"], window[wd["peaklist"]], working_data=wd
    )
    wd = hp.analysis.clean_rr_intervals(working_data=wd, method="quotient-filter")
    rr_list = wd["RR_list_cor"]
    rr_diff = np.diff(rr_list)
    rr_sqdiff = np.power(rr_diff, 2)
    wd, msrs = hp.analysis.calc_ts_measures(
        rr_list, rr_diff, rr_sqdiff, working_data=wd
    )
    wd, msrs = hp.analysis.calc_fd_measures(measures=msrs, working_data=wd)
    bvp_feats = pd.DataFrame([msrs])[
        [
            "bpm",
            "sdnn",
            'rmssd',
            "pnn50",
            "hr_mad",
            "lf",
            "hf",
            "lf/hf",
            "p_total",
            "lf_nu",
            "hf_nu",
        ]
    ]

    return bvp_feats


def get_eda_feats(window, fs):
    signals, info = nk.eda_process(window, fs)
    mean_eda = np.nanmean(window)
    mean_tonic = np.nanmean(signals["EDA_Tonic"])
    scr_peak_count = np.nansum(signals["SCR_Peaks"])
    scr_sum_amp = np.nansum(info["SCR_Amplitude"])
    scr_mean_amp = np.nanmean(info["SCR_Amplitude"])
    scr_mean_risetime = np.nanmean(info["SCR_RiseTime"])
    scr_mean_recoverytime = np.nanmean(info["SCR_RecoveryTime"])

    eda_feats = pd.DataFrame(
        [
            {
                "mean_eda": mean_eda,
                "mean_tonic": mean_tonic,
                "scr_peak_count": scr_peak_count,
                "scr_sum_amp": scr_sum_amp,
                "scr_mean_amp": scr_mean_amp,
                "scr_mean_risetime": scr_mean_risetime,
                "scr_mean_recoverytime": scr_mean_recoverytime,
            }
        ]
    )

    return eda_feats


def process(subjects, bvp_prepdata_df, eda_prepdata_df, bvp_fs, eda_fs, ovlap, winsz):
    feat_mat_list = []
    for subject in subjects:
        # BVP
        bvp_probe = np.array(
            bvp_prepdata_df[bvp_prepdata_df["Subject"] == subject]["valueBVP"]
        )
        bvp_smooth = uniform_filter1d(
            bvp_probe, size=int(0.75 * bvp_fs), mode="nearest"
        )
        bvp_windowed = ctf.timewindowpadded(
            data=bvp_probe, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        smooth_windowed = ctf.timewindowpadded(
            data=bvp_smooth, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        bvp_holder_list = []
        for windnum in range(0,bvp_windowed.shape[0]):
            bvp_feat_row = get_bvp_feats(
                window=bvp_windowed[windnum],
                smooth_window=smooth_windowed[windnum],
                fs=bvp_fs,
            )
            bvp_holder_list.append(bvp_feat_row)
        bvp_holder_df = pd.concat(bvp_holder_list, ignore_index=True)

        # EDA
        eda_probe = np.array(
            eda_prepdata_df[eda_prepdata_df["Subject"] == subject]["valueEDA"]
        )
        eda_windowed = ctf.timewindowpadded(
            data=eda_probe, fs=eda_fs, ovlap=ovlap, winsz=winsz
        )
        eda_holder_list = []
        for windnum in range(0,eda_windowed.shape[0]):
            eda_feat_row = get_eda_feats(window=eda_windowed[windnum], fs=eda_fs)
            eda_holder_list.append(eda_feat_row)
        eda_holder_df = pd.concat(eda_holder_list, ignore_index=True)

        valid_len = min(len(bvp_holder_df), len(eda_holder_df))
        joined_df = pd.concat([bvp_holder_df.iloc[:valid_len], eda_holder_df.iloc[:valid_len]], axis=1)
        joined_df.insert(loc=0, column="Subject", value=[subject] * len(joined_df))
        feat_mat_list.append(joined_df)
        
    feat_mat_df = pd.concat(feat_mat_list, ignore_index=True)
    return feat_mat_df


basal_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=basal_bvp_prepdata,
    eda_prepdata_df=basal_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

tscene_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=tscene_bvp_prepdata,
    eda_prepdata_df=tscene_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

warnings.filterwarnings('ignore')

In [46]:
basal_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,62.466476,60.745158,58.491992,0.136986,39.0625,1.020411e+03,1.069850e+03,0.953789,2.602937e+03,48.817396,51.182604,-0.005557,-0.005470,17,0.133718,0.007866,0.455882,0.750000
1,1,63.840399,74.530595,67.652525,0.126582,46.8750,9.577659e+02,8.816200e+02,1.086370,2.133140e+03,52.069872,47.930128,-0.000625,-0.000620,17,0.109199,0.006825,0.593750,0.950000
2,1,67.908512,101.847943,86.004657,0.121622,62.5000,2.391600e+03,2.279704e+03,1.049084,8.190353e+03,51.197694,48.802306,0.002401,0.002226,18,0.096585,0.005681,1.132353,1.100000
3,1,68.746803,86.251777,62.814230,0.145161,62.5000,8.017548e+02,1.627016e+03,0.492776,2.820222e+03,33.010728,66.989272,0.009608,0.009637,15,0.122990,0.008199,1.050000,1.027778
4,2,71.431793,83.631418,54.698887,0.102041,46.8750,1.013593e+03,8.258658e+02,1.227310,2.730259e+03,55.102794,44.897206,-0.023166,-0.023263,74,0.429396,0.005882,0.414384,0.535714
5,2,74.250000,78.236675,58.271328,0.091837,46.8750,5.973643e+02,8.025838e+02,0.744301,4.640375e+03,42.670461,57.329539,0.001783,0.001801,78,0.461093,0.005911,0.442308,0.632353
6,2,77.111562,85.465727,64.847496,0.102041,62.5000,6.725265e+02,3.094232e+02,2.173484,2.605237e+03,68.488895,31.511105,0.002389,0.001928,26,0.402167,0.015468,0.557692,0.972222
7,2,85.404504,106.789867,111.293637,0.189873,54.6875,2.977515e+03,1.315640e+03,2.263169,6.425113e+03,69.354939,30.645061,-0.000033,0.000183,22,0.376104,0.017096,0.579545,0.966667
8,3,81.923703,56.356810,50.081731,0.128205,31.2500,8.102969e+02,1.374918e+03,0.589342,2.323080e+03,37.080880,62.919120,0.000878,0.000861,35,0.156139,0.004461,0.414286,0.550000
9,3,85.128820,55.584121,51.950588,0.100000,31.2500,5.423301e+02,8.034705e+02,0.674984,1.635130e+03,40.297951,59.702049,-0.000713,-0.000727,36,0.211838,0.005884,0.416667,0.479167


In [47]:
tscene_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,164.365971,132.219350,149.818175,0.303797,85.9375,1296.090196,2059.325665,0.629376,3355.415861,38.626813,61.373187,0.009317,0.009089,20,0.303490,0.015175,0.650000,0.982143
1,1,183.067203,117.753425,163.549030,0.292683,78.1250,665.331751,1607.125991,0.413989,2272.457742,29.278069,70.721931,0.003045,0.003111,26,0.280132,0.010774,0.480769,0.750000
2,1,192.642140,96.986942,118.198472,0.258427,62.5000,114.327087,2356.942880,0.048507,2471.269967,4.626248,95.373752,0.004435,0.004640,16,0.303442,0.018965,0.515625,0.477273
3,1,152.102377,151.589901,139.412634,0.296875,109.3750,985.405568,4644.389513,0.212171,5629.795081,17.503400,82.496600,0.026684,0.026693,5,0.205959,0.041192,0.650000,5.500000
4,2,91.596330,117.800725,146.432282,0.218750,31.2500,757.549784,3482.011957,0.217561,4239.561741,17.868587,82.131413,-0.007472,-0.006858,9,0.306149,0.034017,1.222222,3.000000
5,2,92.530120,129.083685,182.116099,0.245283,39.0625,293.682833,2727.169537,0.107688,3020.852370,9.721853,90.278147,-0.007493,-0.007410,10,0.480404,0.048040,1.475000,2.305556
6,2,103.572519,160.061100,199.674841,0.250000,62.5000,550.521600,5334.521355,0.103200,5885.042955,9.354589,90.645411,-0.003431,-0.004394,27,0.342504,0.013173,0.538462,0.630952
7,2,100.702622,157.484444,177.532013,0.211538,46.8750,1723.003999,5999.462752,0.287193,7722.466751,22.311575,77.688425,-0.023351,-0.023370,20,0.437738,0.021887,0.587500,0.783333
8,3,109.990557,121.862416,115.772710,0.122222,62.5000,3340.782521,6571.043318,0.508410,9911.825839,33.705016,66.294984,-0.021928,-0.018657,6,1.344445,0.224074,1.458333,3.625000
9,3,110.678383,112.241643,115.461179,0.135802,39.0625,1646.535178,4537.992536,0.362833,6184.527714,26.623459,73.376541,-0.013049,-0.012558,15,0.420786,0.028052,0.950000,1.318182


In [48]:
def remove_outliers_iqr(col, ref, p_low=0.25, p_high=0.75, treatment="elim"):
    q1 = ref.quantile(p_low)
    q3 = ref.quantile(p_high)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    if treatment == "impute":
        return col.clip(lower=lower, upper=upper)
    else:
        return col[(col >= lower) & (col <= upper)]


def remove_outliers_zscore(col, ref):
    mean = ref.mean()
    std = ref.std()
    return col[np.abs((col - mean) / std) < 3]


def remove_outliers_winsor(col, p_low=0.05, p_high=0.95, treatment="impute"):
    lower = col.quantile(p_low)
    upper = col.quantile(p_high)
    return col.clip(lower=lower, upper=upper)

channel_cols = [col for col in basal_feat_mat.columns if col not in ["Subject"]]

## Baseline outlier treatment
basal_outlier_treated = basal_feat_mat.copy()

if outliermeth == 'winsor_outlier':
    basal_outlier_treated[channel_cols] = basal_feat_mat[channel_cols].apply(
        lambda s: remove_outliers_winsor(s, p_low=0.05, p_high=0.95)
    )
elif outliermeth == 'zscore_outlier':
    basal_outlier_treated[channel_cols] = basal_feat_mat[channel_cols].apply(
        lambda s: remove_outliers_zscore(s, s)
    )
elif outliermeth == 'quant_outlier':
    basal_outlier_treated[channel_cols] = basal_feat_mat[channel_cols].apply(
        lambda s: remove_outliers_iqr(s, s, p_low=0.05, p_high=0.95, treatment=outliertreatment)
    )


## Target scene outlier treatment
tscene_outlier_treated = tscene_feat_mat.copy()

if outliermeth == 'winsor_outlier':
    tscene_outlier_treated[channel_cols] = tscene_feat_mat[channel_cols].apply(
        lambda s: remove_outliers_winsor(s, p_low=0.1, p_high=0.9)
    )
elif outliermeth == 'zscore_outlier':
    tscene_outlier_treated[channel_cols] = tscene_feat_mat[channel_cols].apply(
        lambda s: remove_outliers_zscore(s, basal_feat_mat[s.name])
    )
elif outliermeth == 'quant_outlier':
    tscene_outlier_treated[channel_cols] = tscene_feat_mat[channel_cols].apply(
        lambda s: remove_outliers_iqr(s, basal_feat_mat[s.name], p_low=0.05, p_high=0.95, treatment=outliertreatment)
    )

basal_outlier_treated.dropna(inplace=True)
tscene_outlier_treated.dropna(inplace=True)

In [49]:
tscene_outlier_treated

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,164.365971,132.219350,149.818175,0.303797,85.9375,1296.090196,2059.325665,0.629376,3355.415861,38.626813,61.373187,0.009317,0.009089,20,0.303490,0.015175,0.650000,0.982143
3,1,152.102377,151.589901,139.412634,0.296875,109.3750,985.405568,4644.389513,0.212171,5629.795081,17.503400,82.496600,0.026684,0.026693,5,0.205959,0.041192,0.650000,5.500000
4,2,91.596330,117.800725,146.432282,0.218750,31.2500,757.549784,3482.011957,0.217561,4239.561741,17.868587,82.131413,-0.007472,-0.006858,9,0.306149,0.034017,1.222222,3.000000
5,2,92.530120,129.083685,182.116099,0.245283,39.0625,293.682833,2727.169537,0.107688,3020.852370,9.721853,90.278147,-0.007493,-0.007410,10,0.480404,0.048040,1.475000,2.305556
6,2,103.572519,160.061100,199.674841,0.250000,62.5000,550.521600,5334.521355,0.103200,5885.042955,9.354589,90.645411,-0.003431,-0.004394,27,0.342504,0.013173,0.538462,0.630952
7,2,100.702622,157.484444,177.532013,0.211538,46.8750,1723.003999,5999.462752,0.287193,7722.466751,22.311575,77.688425,-0.023351,-0.023370,20,0.437738,0.021887,0.587500,0.783333
8,3,109.990557,121.862416,115.772710,0.122222,62.5000,3340.782521,6571.043318,0.508410,9911.825839,33.705016,66.294984,-0.021928,-0.018657,6,1.344445,0.224074,1.458333,3.625000
9,3,110.678383,112.241643,115.461179,0.135802,39.0625,1646.535178,4537.992536,0.362833,6184.527714,26.623459,73.376541,-0.013049,-0.012558,15,0.420786,0.028052,0.950000,1.318182
10,3,109.534426,121.270613,142.400373,0.116279,31.2500,1288.992168,3400.934865,0.379011,4689.927033,27.484269,72.515731,-0.050638,-0.051721,8,0.969782,0.121223,2.031250,2.178571
11,3,109.040675,139.240327,165.975173,0.150685,54.6875,1112.296061,3621.034288,0.307176,4733.330349,23.499227,76.500773,0.164744,0.165153,8,1.390860,0.173857,1.906250,2.321429


In [50]:
list(tscene_outlier_treated['Subject'].unique())

[1, 2, 3, 4, 6, 8, 9]

In [51]:
def norm_with_ref(
    subjects,
    feat_mat,
    ref_mat,
    normtype=str,
):
    channel_cols = [col for col in ref_mat.columns if col not in ["Subject"]]

    normdata_list = []

    # Data is normalized per subject 
    for subject in subjects:
        data = feat_mat[feat_mat["Subject"] == subject].copy()

        ref_data = ref_mat[ref_mat["Subject"] == subject].copy().drop("Subject", axis=1)


        if normtype == "zscore":
            scaler = StandardScaler()
            scaler.fit(ref_data[channel_cols])
            data[channel_cols] = scaler.transform(data[channel_cols])
        elif normtype == "minmax":
            scaler = MinMaxScaler()
            scaler.fit(ref_data[channel_cols])
            data[channel_cols] = scaler.transform(data[channel_cols])
        normdata_list.append(data.dropna())
    normdata_df = pd.concat(normdata_list, ignore_index=True)

    return normdata_df

subjects = tscene_outlier_treated['Subject'].unique()
norm = norm_with_ref(
    subjects=subjects,
    feat_mat=tscene_outlier_treated,
    ref_mat=basal_outlier_treated,
    normtype=normtype,
)
norm

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,16.225190,1.738914,3.319423,7.739100,2.000000,0.310933,0.842371,0.230123,0.201789,0.294666,0.705334,0.980858,0.963736,1.666667,5.572079,3.770316,0.286957,0.663265
1,1,14.272490,2.210185,2.941214,7.445023,3.000000,0.115515,2.691375,-0.472722,0.577271,-0.813642,1.813642,2.126065,2.129026,-3.333333,2.945486,14.103483,0.286957,13.571429
2,2,1.443137,1.385626,1.620882,1.294549,-1.000000,0.067301,3.152987,-0.346798,0.427848,-0.929449,1.929449,0.614132,0.651217,-0.232143,-0.823109,2.508978,4.891193,5.645455
3,2,1.509967,1.780782,2.251396,1.565193,-0.500000,-0.127589,2.402809,-0.419137,0.108803,-1.234748,2.234748,0.613287,0.629306,-0.214286,1.227216,3.759595,6.421678,4.054545
4,2,2.300250,2.865684,2.561650,1.613307,1.000000,-0.019681,4.994051,-0.422092,0.858616,-1.248511,2.248511,0.772247,0.749015,0.089286,-0.395353,0.650205,0.751251,0.218182
5,2,2.094857,2.775443,2.170398,1.220989,0.000000,0.472928,5.654885,-0.300954,1.339632,-0.762949,1.762949,-0.007223,-0.004275,-0.035714,0.725198,1.427275,1.048162,0.567273
6,3,0.946438,0.545311,0.611674,0.641975,1.000000,2.070918,1.694975,0.294472,1.568395,0.326991,0.673009,-3.855830,-3.179704,-0.888889,12.573426,74.282848,7.959571,10.170984
7,3,0.969632,0.466155,0.608773,1.034294,0.250000,0.817137,1.097502,-0.322117,0.862089,-0.395896,1.395896,-2.192904,-2.058288,-0.388889,2.800215,7.979604,4.084158,2.712671
8,3,0.931057,0.540441,0.859614,0.470284,0.000000,0.552547,0.763343,-0.253596,0.578870,-0.308025,1.308025,-9.232387,-9.259375,-0.777778,8.609130,39.493953,12.327351,5.494449
9,3,0.914407,0.688289,1.079128,1.464231,0.750000,0.421788,0.828026,-0.557853,0.587094,-0.714819,1.714819,31.103080,30.618416,-0.777778,13.064536,57.297327,11.374381,5.956329


In [52]:
outpath = os.path.join(base_dir, 'Outputs', f'outlier_{outliertreatment}', outliermeth)
basal_outlier_treated.to_csv(os.path.join(outpath,'non-normalized',rf'biometric_feat_mat_scene_{basalscene}_nonnorm.csv'), index=False)
tscene_outlier_treated.to_csv(os.path.join(outpath,'non-normalized',rf'biometric_feat_mat_scene_{targetscene}_nonnorm.csv'), index=False)
norm.to_csv(os.path.join(outpath, rf'biometric_feat_mat_scene_{targetscene}_{normtype}.csv'), index=False)

In [58]:
x_labels = ["Baseline"] + [f"Scene {i+1}" for i in range(5)]
x_labels

['Baseline', 'Scene 1', 'Scene 2', 'Scene 3', 'Scene 4', 'Scene 5']

In [59]:
x_labels

['Baseline', 'Scene 1', 'Scene 2', 'Scene 3', 'Scene 4', 'Scene 5']

In [60]:
print(x_labels)

['Baseline', 'Scene 1', 'Scene 2', 'Scene 3', 'Scene 4', 'Scene 5']
